# IMPORT

In [7]:
import pandas as pd 
import numpy as np 
import math as math
import joblib
from joblib import dump
import os
import time
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# DATA TEST

In [2]:
def load_Dataset(baseFile, fold_Number): 
    rnmColData = ['user_id', 'item_id', 'rating', 'timestamp']
    test_File = f"{baseFile}/u{fold_Number}.test"
    test_Data = pd.read_csv(test_File, sep="\t", header=None, names=rnmColData)
    test_Data = test_Data.drop(columns=["timestamp"])
    return test_Data

In [3]:
call_base = "../ml-100k"
testdata = load_Dataset(call_base, 1)
testdata


,user_id,item_id,rating
0,1,6,5
1,1,10,3
2,1,12,5
3,1,14,5
4,1,17,3
...,...,...,...
19995,458,648,4
19996,458,1101,4
19997,459,934,3
19998,460,10,3


In [4]:
def GetUniqueUserItemShape(ratingData, fold_num=1):
    max_user = ratingData['user_id'].max()
    unique_user_count = ratingData['user_id'].nunique()
    unique_item_count = ratingData['item_id'].nunique()
    # sparasity
    sparsity = 1 - (len(ratingData) / (max_user * unique_item_count))
    result = f"fold {fold_num}, uniq user =  {unique_user_count},uniq item = {unique_item_count},max user = {max_user}, sparsity = {sparsity:.4f}"
    return result

# Loop untuk fold 1 sampai 5
for fold in range(1, 6):
    testdata = load_Dataset(call_base, fold)
    output = GetUniqueUserItemShape(testdata, fold)
    print(output)


fold 1, uniq user =  459,uniq item = 1410,max user = 462, sparsity = 0.9693
fold 2, uniq user =  653,uniq item = 1420,max user = 658, sparsity = 0.9786
fold 3, uniq user =  869,uniq item = 1423,max user = 877, sparsity = 0.9840
fold 4, uniq user =  923,uniq item = 1394,max user = 943, sparsity = 0.9848
fold 5, uniq user =  927,uniq item = 1407,max user = 943, sparsity = 0.9849


In [5]:
def uniqDataTest(ratingData):
    unique_users = sorted(ratingData['user_id'].unique())
    unique_items = sorted(ratingData['item_id'].unique())

    # Grup item_id per user_id jadi list item (ground truth)
    grouped = ratingData.groupby('user_id')['item_id'].apply(list)

    # Buat list groundTruth (list item per user) sesuai urutan unique_users
    groundTruth = [grouped[user] for user in unique_users]

    return groundTruth


# TOY DATA

In [6]:
# toy data ndcg
groundTruthFold1 = [[1, 2, 3, 4, 11], [1, 12, 4, 7], [4, 11, 12, 32], [2, 3, 5, 10]]
groundTruthFold2 = [[3, 5, 7, 10, 11], [2, 3, 5, 11], [1, 10, 11, 12], [4, 7, 12, 21]]
TopNrekFold1 = [[19, 5, 2, 3, 15], [1,4,6,2], [11, 4, 32, 1], [2, 4, 10, 45]]
TopNrekFold2 = [[2, 3, 4, 15, 5], [3,2,12,42], [1, 12, 2, 22], [4, 7, 12, 22]]


# PRECISION

In [7]:
def Precision(groundTruth, TopN, max_N):
    rumusPrecision = len(np.intersect1d(TopN[:max_N], groundTruth)) / max_N
    # print(len(np.intersect1d(TopN[:max_N], groundTruth)))
    # print((np.intersect1d(TopN[:max_N], groundTruth)))
    # print(rumusPrecision)
    return rumusPrecision

In [13]:
def PrecisionLoop(groundTruth, TopN, max_N):
    precisionList = []
    for i in range(len(groundTruth)):
        user_precision = []
        # print(f'\n user ke-{i+1} : ')
        for N in range(1, max_N + 1):
            precision = Precision(groundTruth[i], TopN[i], N)
            user_precision.append(precision)
            # print(f'Precision@{N} = {precision}')
        precisionList.append(user_precision)

    # membuat df
    columns = [f'Precision@{N}' for N in range(1, max_N + 1)]
    precision_df = pd.DataFrame(precisionList, columns=columns)
    # menambhakan index setiap user
    precision_df.index = [f'User {i+1}' for i in range(len(groundTruth))]

    # tambhakn rata-rata precision@n
    precision_df.loc['Rata-Rata'] = precision_df.mean()
    # print(precision_df)
    return precision_df

In [14]:
toyDataTestHitungan = PrecisionLoop(groundTruthFold1, TopNrekFold1, 4)
toyDataTestHitungan

,Precision@1,Precision@2,Precision@3,Precision@4
User 1,0.00,0.000,0.333333,0.5000
User 2,1.00,1.000,0.666667,0.5000
User 3,1.00,1.000,1.000000,0.7500
User 4,1.00,0.500,0.666667,0.5000
Rata-Rata,0.75,0.625,0.666667,0.5625


# AP

In [30]:
def AP(groundTruth, TopN, max_N):
    aPrecision = np.array([Precision(groundTruth, TopN, max_N=x) for x in range(1, max_N+1)])
    # print(f'ini adalah preceison {aPrecision}')
    cekGT = np.array([1 if tp in groundTruth else 0 for tp in TopN[:max_N]])
    # print(f'ini cek gt {cekGT}')
    rumusAP = (1/len(groundTruth) * np.sum(aPrecision * cekGT))
    # print(np.sum(aPrecision * cekGT))
    # print((len(groundTruth)))
    return rumusAP

In [31]:
def APLoop(groundTruth, TopN, max_N):
    apList = []
    for i in range(len(groundTruth)):
        user_AP = []
        for N in range(1, max_N + 1):
            ap = AP(groundTruth[i], TopN[i], N)
            user_AP.append(ap)
            # print(f'AP@{N} = {ap}')
        apList.append(user_AP)
    
    # membuat df
    columns = [f'AP@{N}' for N in range(1, max_N + 1)]
    ap_df = pd.DataFrame(apList, columns=columns)
    ap_df.index = [f'User {i+1}' for i in range(len(groundTruth))]

    # tambhakan rata-rata AP
    ap_df.loc['Rata-Rata'] = ap_df.mean()
    # print(ap_df)
    return ap_df

In [34]:
calApLoppTestHitungan = APLoop(groundTruthFold1, TopNrekFold1, 4)
calApLoppTestHitungan

,AP@1,AP@2,AP@3,AP@4
User 1,0.0000,0.0000,0.066667,0.166667
User 2,0.2500,0.5000,0.500000,0.500000
User 3,0.2500,0.5000,0.750000,0.750000
User 4,0.2500,0.2500,0.416667,0.416667
Rata-Rata,0.1875,0.3125,0.433333,0.458333


# RECALL

In [ ]:
def Recall(groundTruth, TopN, max_N):
    rumusRecall = len(np.intersect1d(TopN[:max_N], groundTruth)) / len(set(groundTruth))
    # print(len(np.intersect1d(TopN[:N], GT)))
    # print(len(set(GT)))
    # print(rumusRecall)
    return rumusRecall

In [ ]:
def RecallLopp(groundTruth, TopN, max_N):
    recallList = []
    for i in range(len(groundTruth)):
        user_recall = []
        # print(f'\n user ke-{i+1} : ')
        for N in range(1, max_N + 1):
            recall = Recall(groundTruth[i], TopN[i], N)
            user_recall.append(recall)
            # print(f'Recall@{N} = {recall}')
        recallList.append(user_recall)

    # membuat df
    columns = [f'Recall@{N}' for N in range(1, max_N + 1)]
    recall_df = pd.DataFrame(recallList, columns=columns)
    # menambhakan index setiap user
    recall_df.index = [f'User {i+1}' for i in range(len(groundTruth))]

    # tambhakan rata-rata Recall@n
    recall_df.loc['Rata-Rata'] = recall_df.mean()
    # print(recall_df)
    return recall_df

# F1SCORE

In [ ]:
def F1Score(groundTruth, TopN, max_N):
    precision = Precision(groundTruth, TopN, max_N)
    recall = Recall(groundTruth, TopN, max_N)
    rumusF1Score = ((2 * precision * recall) / (precision + recall)) 
    if precision > 0 and recall > 0: 
        rumusF1Score = rumusF1Score
    else :
        rumusF1Score = 0
    # print(rumusF1Score)
    return rumusF1Score
    

In [ ]:
def F1ScoreLoop(groundTruth, TopN, max_N):
    f1List = []
    for i in range(len(groundTruth)):
        user_f1 = []
        # print(f'\n user ke-{i+1} : ')
        for N in range(1, max_N + 1):
            f1 = F1Score(groundTruth[i], TopN[i], N)
            user_f1.append(f1)
            # print(f'F1Score@{N} = {f1}')
        f1List.append(user_f1)

    # membuat df
    columns = [f'F1Score@{N}' for N in range(1, max_N + 1)]
    f1_df = pd.DataFrame(f1List, columns=columns)
    # menambhakan index setiap user
    f1_df.index = [f'User {i+1}' for i in range(len(groundTruth))]

    # tambhakan rata-rata F1Score@n
    f1_df.loc['Rata-Rata'] = f1_df.mean()
    # print(f1_df)
    return f1_df

# LOOP REAL DATA

In [ ]:
import os
import joblib

def MetricLoop(base_input_dir, base_output_dir, dataset_dir, metric_func, metric_name, N=20):
    """
    Menghitung metrik evaluasi secara dinamis untuk setiap fold dan menyimpan hasilnya.
    
    Parameters:
        base_input_dir (str): Direktori input per fold
        base_output_dir (str): Direktori output per fold
        dataset_dir (str): Path ke dataset
        metric_func (function): Fungsi metrik yang akan dipanggil, misal: AP, NDCG, dll
        metric_name (str): Nama metrik (untuk nama file hasil)
        N (int): Top-N rekomendasi yang dievaluasi
    """

    for fold in range(1, 6):
        print(f'\nProcessing fold {fold}...')

        try:
            # Load dan siapkan ground truth
            testdata = load_Dataset(dataset_dir, fold)
            dataTestUniq = uniqDataTest(testdata)

            fold_input_path = os.path.join(base_input_dir, str(fold))
            fold_output_path = os.path.join(base_output_dir, str(fold))
            os.makedirs(fold_output_path, exist_ok=True)

            # Loop semua file joblib (model output) di dalam fold
            for filename in os.listdir(fold_input_path):
                if filename.endswith('.joblib'):
                    filepath = os.path.join(fold_input_path, filename)
                    print(f'Processing file: {filename}')

                    try:
                        openFungsMetrik = joblib.load(filepath)

                        # Hitung metrik
                        result_df = metric_func(dataTestUniq, openFungsMetrik, N)

                        # Simpan hasil joblib
                        result_filename = filename.replace('.joblib', f'_{metric_name}.joblib')
                        result_path = os.path.join(fold_output_path, result_filename)
                        joblib.dump(result_df, result_path)
                        print(f'Saved joblib to: {result_path}')

                        # Simpan hasil Excel
                        # excel_filename = filename.replace('.joblib', f'_{metric_name}.xlsx')
                        # excel_path = os.path.join(fold_output_path, excel_filename)
                        # result_df.to_excel(excel_path)
                        # print(f'Saved Excel to: {excel_path}')

                    except Exception as e:
                        print(f'Error processing {filename}: {e}')

        except Exception as e:
            print(f'Error in fold {fold}: {e}')


## AP

### Jac

In [44]:
# joblib.load('../case/topN/Jac/topNuser/1/5_20_0.7.joblib')

In [6]:
# JacProsesAPUser = MetricLoop(
#     base_input_dir='../case/topN/Jac/topNUser',
#     base_output_dir='../evaluasiTambahan/AP/Jac/user',
#     dataset_dir='../ml-100k',
#     metric_func=APLoop,
#     metric_name='AP',
#     N=20
# )


In [5]:
# JacProsesAPItem = MetricLoop(
#     base_input_dir='../case/topN/Jac/topNItem',
#     base_output_dir='../evaluasiTambahan/AP/Jac/item',
#     dataset_dir='../ml-100k',
#     metric_func=APLoop,
#     metric_name='AP',
#     N=20
# )

### RJ

In [4]:
# RJProsesAPUser = MetricLoop(
#     base_input_dir='../case/topN/RJ2/topNUser',
#     base_output_dir='../evaluasiTambahan/AP/RJ/user',
#     dataset_dir='../ml-100k',
#     metric_func=APLoop,
#     metric_name='AP',
#     N=20
# )

In [3]:
# RJProsesAPItem = MetricLoop(
#     base_input_dir='../case/topN/RJ2/topNItem',
#     base_output_dir='../evaluasiTambahan/AP/RJ/item',
#     dataset_dir='../ml-100k',
#     metric_func=APLoop,
#     metric_name='AP',
#     N=20
# )

### HYBRID

In [2]:
# calProsesHybridAP = MetricLoop(
#     base_input_dir='../case/topN/Jac/topNHybrid',
#     base_output_dir='../evaluasiTambahan/AP/Jac/hybrid',
#     dataset_dir='../ml-100k',
#     metric_func=APLoop,
#     metric_name='AP',
#     N=20
# )

In [1]:
# calProsesHybridAP = MetricLoop(
#     base_input_dir='../case/topN/RJ2/topNHybrid',
#     base_output_dir='../evaluasiTambahan/AP/RJ/hybrid',
#     dataset_dir='../ml-100k',
#     metric_func=APLoop,
#     metric_name='AP',
#     N=20
# )

### HYBRID TERBAIK

## RECALL

### Jac

In [ ]:
JacProsesRecallUser = MetricLoop(
    base_input_dir='../case/topN/Jac/topNUser',
    base_output_dir='../evaluasiTambahan/Recall/Jac/user',
    dataset_dir='../ml-100k',
    metric_func=RecallLopp,
    metric_name='Recall',
    N=20
)

In [ ]:
JacProsesRecallItem = MetricLoop(
    base_input_dir='../case/topN/Jac/topNItem',
    base_output_dir='../evaluasiTambahan/Recall/Jac/item',
    dataset_dir='../ml-100k',
    metric_func=RecallLopp,
    metric_name='Recall',
    N=20
)

### RJ

In [ ]:
RJProsesRecallUser = MetricLoop(
    base_input_dir='../case/topN/RJ2/topNUser',
    base_output_dir='../evaluasiTambahan/Recall/RJ/user',
    dataset_dir='../ml-100k',
    metric_func=RecallLopp,
    metric_name='Recall',
    N=20
)

In [ ]:
RJProsesRecallItem = MetricLoop(
    base_input_dir='../case/topN/RJ2/topNItem',
    base_output_dir='../evaluasiTambahan/Recall/RJ/item',
    dataset_dir='../ml-100k',
    metric_func=RecallLopp,
    metric_name='Recall',
    N=20
)

### HYBRID

In [ ]:
JacProsesHybrdRecall = MetricLoop(
    base_input_dir='../case/topN/Jac/topNHybrid',
    base_output_dir='../evaluasiTambahan/Recall/Jac/hybrid',
    dataset_dir='../ml-100k',
    metric_func=RecallLopp,
    metric_name='Recall',
    N=20
)

In [ ]:
RJProsesHybrdRecall = MetricLoop(
    base_input_dir='../case/topN/RJ2/topNHybrid',
    base_output_dir='../evaluasiTambahan/Recall/RJ/hybrid',
    dataset_dir='../ml-100k',
    metric_func=RecallLopp,
    metric_name='Recall',
    N=20
)

### HYBRID TERBAIK

## F1Score

### Jac

### RJ

### HYBRID

### HYBRID TERBAIK

# PENCARIAN USER, ITEM, HYBRID

In [72]:
joblib.load('../evaluasiTambahan/AP/Jac/user/1/5_20_0.7_AP.joblib')

,AP@1,AP@2,AP@3,AP@4,AP@5,AP@6,AP@7,AP@8,AP@9,AP@10,AP@11,AP@12,AP@13,AP@14,AP@15,AP@16,AP@17,AP@18,AP@19,AP@20
User 1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
User 2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
User 3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
User 4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
User 5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
User 456,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
User 457,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
User 458,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
User 459,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [75]:
def evaluate_ap20_across_folds(
    values, 
    folds, 
    path_template, 
    value_position="middle", 
    highlight_best=True, 
    round_digits=4
):
    data = {}

    for val in values:
        ap_per_fold = []
        for fold in folds:
            # Format file path
            if value_position == "start":
                filename = f"{val}_20_0.7_AP.joblib"
            elif value_position == "middle":
                filename = f"5_{val}_0.7_AP.joblib"
            elif value_position == "end":
                filename = f"5_5_{val}_AP.joblib"
            else:
                raise ValueError("value_position harus salah satu dari: 'start', 'middle', 'end'.")

            filepath = path_template.format(fold=fold, val=filename)

            df = joblib.load(filepath)
            # AMbil nilai NDCG@20 dari DataFrame
            ap20 = df.loc["Rata-Rata", "AP@20"]
            ap_per_fold.append(ap20)
        data[val] = ap_per_fold
    # Buat DataFrame dari dictionary
    ap_df = pd.DataFrame.from_dict(data, orient="index", 
                                     columns=[f"Fold-{i}" for i in folds])
    ap_df = ap_df.round(round_digits)
    ap_df["Rata-Rata"] = ap_df.mean(axis=1).round(round_digits)
    ap_df.reset_index(inplace=True)
    ap_df.rename(columns={"index": "value"}, inplace=True)

    if highlight_best:
        max_value = ap_df["Rata-Rata"].max()
        ap_df["terbaik"] = ap_df["Rata-Rata"] == max_value

    return ap_df


In [70]:
values = [5, 10, 15, 18, 20, 25, 30, 40, 50, 100, 200]
folds = [1, 2, 3, 4, 5]

In [80]:
pathTemplate = '../evaluasiTambahan/AP/Jac/item/{fold}/{val}'
APUser20 = evaluate_ap20_across_folds(
    values, 
    folds, 
    pathTemplate, 
    value_position="middle", 
    highlight_best=True, 
    round_digits=4
)


APUser20

,value,Fold-1,Fold-2,Fold-3,Fold-4,Fold-5,Rata-Rata,terbaik
0,5,0.0014,0.0015,0.0016,0.0014,0.0011,0.0014,True
1,10,0.0013,0.0015,0.0015,0.0014,0.0010,0.0013,False
2,15,0.0013,0.0015,0.0015,0.0014,0.0008,0.0013,False
3,18,0.0012,0.0015,0.0015,0.0013,0.0009,0.0013,False
4,20,0.0012,0.0015,0.0015,0.0014,0.0009,0.0013,False
5,25,0.0013,0.0015,0.0015,0.0014,0.0008,0.0013,False
6,30,0.0012,0.0015,0.0015,0.0014,0.0008,0.0013,False
7,40,0.0012,0.0015,0.0015,0.0014,0.0008,0.0013,False
8,50,0.0012,0.0015,0.0015,0.0013,0.0008,0.0013,False
9,100,0.0012,0.0015,0.0015,0.0014,0.0008,0.0013,False
